# Clase 6 — La corrida completa con nikodym
## Modelador de Riesgo de Crédito · Academia Bayes

**Banco Austral · datos 100% sintéticos · demo en vivo.** Este notebook toma la matriz fabricada
en C2, ejecuta el pipeline F1 de punta a punta y deja la evidencia que un tercero necesita para
reproducir, revisar y gobernar el modelo.

**Cómo usarlo:** se puede leer sin ejecutar porque conserva todas las salidas. Para reproducirlo,
elige *Entorno de ejecución → Ejecutar todas* y recorre las celdas en orden. Cada paso dice qué hace,
qué salida esperar y cómo interpretarla.

**Glosario mínimo:** DEV ajusta el modelo; HO lo prueba sin usar meses futuros; OOT prueba meses
posteriores; TTD son solicitudes recientes todavía sin desempeño; TC es la tasa de malos de largo
plazo; WoE traduce cada tramo de una variable a evidencia de riesgo.

**Dos configuraciones, dos propósitos:** `config` es la receta real y es la única que alimenta el
modelo final. `config_roto` aparece sólo en el paso 2: es una copia descartable a la que agregamos
un error intencional para comprobar que el contrato detiene datos inválidos. Nunca se mezclan.

| Paso | Del dato al expediente | Salida visible |
|---|---|---|
| 0–3 | versión, matriz, contrato y corrida | `config_hash`, error real y `run_id` |
| 4–6 | objetos de comité y trazabilidad | scorecard, métricas, trail y dos ataques |
| 7–9 | identidad y documentos | lineage, model card, informe y resumen |

> La herramienta automatiza cálculo y evidencia. La recomendación, los límites de uso y los
> gatillos siguen siendo responsabilidad del modelador.

## Paso 0 · Fijar el entorno — 2 min

El pin evita que Colab instale una versión distinta de la que fue validada para la clase.
`[scoring]` pide además las dependencias opcionales del motor de scorecards. Todo lo que sigue usa
archivos locales del runtime de Colab: no requiere montar Drive ni clonar este repositorio.
**Qué mirar:** la celda debe imprimir `nikodym 1.11.0`. Si muestra otra versión, se detiene la
revisión porque ya no es el entorno con que se comprobaron estos resultados.

In [1]:
!pip install -q "nikodym[scoring]==1.11.0"

import copy
import hashlib
import json
import time
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import nikodym

from nikodym.audit import read_trail
from nikodym.calibration import CalibrationConfig
from nikodym import check_dataset
from nikodym.core.config import NikodymConfig, config_hash
from nikodym.data import DataConfig
from nikodym.report import ReportConfig
from nikodym.ui.presets import standard_preset

PIN_NIKODYM = "1.11.0"
URL_MATRIZ = "https://raw.githubusercontent.com/nexolabs-gh/datos-riesgo-credito/main/austral_matriz_modelable.parquet"
RUTA_MATRIZ = "austral_matriz_modelable.parquet"
SALIDA = Path("salida_c6")
TRAIL = SALIDA / "audit_trail.jsonl"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)
assert nikodym.__version__ == PIN_NIKODYM
print("nikodym", nikodym.__version__)

nikodym 1.11.0


## Paso 1 · Matriz por URL y configuración F1 — 6 min

La descarga no modela: sólo vuelve local una matriz con índice, target nullable, partición y 108
candidatas. Después se construye la receta en cuatro capas:

1. `ESQUEMA`, `PREDICTORAS` y `CATEGORICAS` describen qué entra y con qué tipo. Se leen **de la
   matriz misma** con `esquema_de`: una columna, un tipo y si admite nulos — el contrato que
   producción va a exigir, sin escribirlo a mano.
2. `standard_preset()` aporta el orden estándar del pipeline F1; `deepcopy` permite editar una copia.
3. `cfg` es el diccionario editable donde declaramos target, muestras, selección, calibración y gobierno.
4. `config` es ese diccionario ya validado por nikodym; sólo este objeto entra a la corrida final.

**Qué mirar:** el `config_hash` identifica la receta completa. Si cambia una decisión del config,
cambia el hash aunque el archivo de datos sea el mismo.

In [2]:
# 1) La fuente es pública y reproducible: Colab descarga la misma matriz que usamos aquí.
urllib.request.urlretrieve(URL_MATRIZ, RUTA_MATRIZ)
matriz = pd.read_parquet(RUTA_MATRIZ)
METADATOS = {"id_cliente", "fecha_solicitud", "cohorte", "muestra", "malo", "indeterminado"}


def esquema_de(matriz):
    """El contrato de entrada, leído de la matriz: nombre, tipo y si admite nulos."""
    def tipo(serie):
        if serie.name == "malo" or pd.api.types.is_float_dtype(serie):
            return "float"
        if pd.api.types.is_integer_dtype(serie):
            return "int"
        return "bool" if pd.api.types.is_bool_dtype(serie) else "str"
    esquema = []
    for columna in matriz.columns:
        regla = {"name": columna, "dtype": tipo(matriz[columna])}
        if columna == "malo" or columna not in METADATOS:
            regla["nullable"] = True          # el target y las candidatas pueden venir vacíos
        esquema.append(regla)
    predictoras = [c for c in matriz.columns if c not in METADATOS]
    categoricas = [c for c in predictoras if pd.api.types.is_object_dtype(matriz[c])]
    return esquema, predictoras, categoricas


# 2) El contrato de entrada: qué columnas entran y con qué tipo (no es un resultado del modelo).
ESQUEMA, PREDICTORAS, CATEGORICAS = esquema_de(matriz)

# 3) Partimos de la receta F1 y reemplazamos sus decisiones por las del caso Austral.
cfg = copy.deepcopy(standard_preset()["config"])
cfg["data"]["load"]["source"] = RUTA_MATRIZ
cfg["data"]["schema"]["columns"] = ESQUEMA
cfg["data"]["schema"]["index_col"] = "id_solicitud"

# Target y elegibilidad: 1 es malo; los NaN son TTD sin desempeño y no ajustan el modelo.
cfg["data"]["target"]["bad_rule"] = {
    "all_of": [{"col": "malo", "op": "==", "value": 1}]
}
cfg["data"]["target"]["good_rule"] = None
cfg["data"]["target"]["indeterminate_rule"] = {
    "all_of": [{"col": "indeterminado", "op": "==", "value": True}]
}
cfg["data"]["target"]["exclusion_rules"] = [{
    "name": "sin_desempeno",
    "rule": {"all_of": [{"col": "malo", "op": "isna"}]},
}]
cfg["data"]["partition"]["ttd_includes_excluded"] = True

# Partición temporal: OOT son los meses futuros; el resto elegible se divide DEV/HO.
cfg["data"]["partition"]["strategy"] = {
    "type": "cohort",
    "cohort_col": "cohorte",
    "oot_cohorts": ["2025-03", "2025-04", "2025-05", "2025-06"],
    "holdout_fraction": 0.3,
}

# Selección: primero IV mínimo y después control de redundancia por correlación.
cfg["binning"]["feature_columns"] = PREDICTORAS
cfg["binning"]["categorical_columns"] = CATEGORICAS
cfg["selection"]["min_iv"] = 0.10
cfg["selection"]["correlation"]["threshold"] = 0.70

# Calibración: el nivel central de PD viene de la tasa histórica de largo plazo.
cfg["calibration"]["target_pd"] = 0.054215
cfg["calibration"]["anchor_source"] = "historical_default_rate"
cfg["report"]["formats"] = ["html", "md"]
cfg["report"]["output_dir"] = str(SALIDA)

# Evitamos mezclar el trail de esta ejecución con uno anterior del mismo runtime.
SALIDA.mkdir(exist_ok=True)
TRAIL.unlink(missing_ok=True)
cfg["audit"] = {
    "enabled": True,
    "trail_filename": str(TRAIL),
    "capture_environment": True,
}
cfg["governance"] = {
    "model_name": "austral-scorecard-consumo",
    "cartera": "Consumo Banco Austral",
    "motor": "scoring",
    "fase": "F1",
    "estado_validacion": "en_validacion",
    "author": "Equipo de Modelos — Banco Austral",
    "purpose": "Scorecard de admisión para crédito de consumo, cutoff 560.",
    "assumptions": [
        "Default = 90+ DPD en 12 meses.",
        "Solicitudes 2024-07 a 2025-06; TTD desde 2025-07.",
    ],
    "limitations": ["Modelado sobre aprobados: no observa las rechazadas."],
    "review_period_months": 12,
}

# 4) Validamos primero tres secciones legibles y luego el contrato integral.
#    Desde aquí, `config` es la receta real; `cfg` queda como representación editable.
DataConfig.model_validate(cfg["data"])
CalibrationConfig.model_validate(cfg["calibration"])
ReportConfig.model_validate(cfg["report"])
config = NikodymConfig.model_validate(cfg)
config_hash_actual = config_hash(config)

print(f"matriz: {len(matriz):,} filas · {len(PREDICTORAS)} candidatas "
      f"({len(CATEGORICAS)} categóricas) · índice {matriz.index.name}")
print("target: malo nullable · indeterminados fuera del ajuste · TTD excluida del ajuste")
print(f"config_hash: {config_hash_actual}")

matriz: 15,665 filas · 108 candidatas (7 categóricas) · índice id_solicitud
target: malo nullable · indeterminados fuera del ajuste · TTD excluida del ajuste
config_hash: 8bdf3d7bad0a37e4871f5de7fd8fef9c47ba55f199fa52435e9b3cfc13f7a9ba


## Paso 2 · El contrato avisa y también bloquea — 5 min

`check_dataset` avisa si el esquema y las columnas no calzan. Una regla de rango sí bloquea la
corrida: aquí se impone deliberadamente `antiguedad_meses ≤ 344`, límite que la matriz viola.

**Ésta es una prueba controlada, no otro modelo.** `cfg_roto = deepcopy(cfg)` crea una copia
independiente; la regla falsa se agrega sólo allí y se apagan informe, auditoría y gobierno para no
dejar documentos de una corrida que sabemos inválida. `cfg` y `config` permanecen intactos.

`nikodym.run` no lanza una excepción: devuelve un `Study` con estado `failed` y el error adentro
(`run_context.error`), para que el rechazo quede registrado igual que en producción.

**Qué mirar:** `compatible: True` valida nombres y tipos; el error posterior prueba que una regla
de negocio incumplida detiene la corrida antes de estimar el modelo.

In [3]:
# Primera puerta: ¿están las columnas y el índice que exige el contrato?
columnas_parquet = pq.read_schema(RUTA_MATRIZ).names
revision = check_dataset(
    config,
    columns=[columna for columna in columnas_parquet if columna != "id_solicitud"],
    index_columns=["id_solicitud"],
)
print("check_dataset compatible:", revision.compatible)
print("mismatches:", revision.mismatches)

# Segunda puerta: ensayamos un error de contenido en una COPIA descartable.
cfg_roto = copy.deepcopy(cfg)
regla_antiguedad = next(
    regla for regla in cfg_roto["data"]["schema"]["columns"]
    if regla["name"] == "antiguedad_meses"
)
regla_antiguedad["le"] = 344
cfg_roto["audit"] = None
cfg_roto["governance"] = None
cfg_roto["report"] = None
config_roto = NikodymConfig.model_validate(cfg_roto)

# La corrida rota no lanza una excepción: devuelve un Study fallido con el error registrado.
estudio_roto = nikodym.run(config_roto)
assert estudio_roto.run_context.status == "failed"
assert estudio_roto.run_context.error.type == "DataValidationError"

print(f"\n✅ El contrato detuvo la corrida antes del binning (status {estudio_roto.run_context.status}):")
lineas_error = estudio_roto.run_context.error.message.splitlines()
print("\n".join(lineas_error[:5]))
print(f"… {len(lineas_error) - 5} hallazgos adicionales omitidos de esta vista.")
print("LECTURA: las columnas son compatibles, pero una regla de rango deliberadamente falsa bloquea el modelo.")

check_dataset compatible: True
mismatches: ()

✅ El contrato detuvo la corrida antes del binning (status failed):
El dataset no cumple el esquema declarado en data.schema (331 problemas):
- «antiguedad_meses»: es mayor que el máximo declarado (valor: 345, fila S0013564).
- «antiguedad_meses»: es mayor que el máximo declarado (valor: 355, fila S0025773).
- «antiguedad_meses»: es mayor que el máximo declarado (valor: 352, fila S0025557).
- «antiguedad_meses»: es mayor que el máximo declarado (valor: 355, fila S0025548).
… 327 hallazgos adicionales omitidos de esta vista.
LECTURA: las columnas son compatibles, pero una regla de rango deliberadamente falsa bloquea el modelo.


## Paso 3 · La corrida completa — 6 min

Carga → esquema → target → partición → binning/WoE → selección → modelo → scorecard →
calibración → desempeño/estabilidad → validación → informe. **Qué mirar:** `status done` demuestra
que toda la cadena terminó; no demuestra que el modelo sea bueno. El `run_id` identifica esta
ejecución y debe acompañar sus documentos.

> **Atención:** esta celda usa `config`, la receta real validada en el paso 1. No usa
> `config_roto`, que terminó su función pedagógica en el paso anterior.

In [4]:
# Única estimación del modelo válido: la prueba rota anterior no alimenta esta corrida.
inicio = time.perf_counter()
study = nikodym.run(config)
segundos_corrida = time.perf_counter() - inicio

assert study.run_context.status == "done", study.run_context.error
run_id = study.run_context.run_id
print(f"corrida completa ✅ · {segundos_corrida:.1f} s")
print("run_id:", run_id)
print("LECTURA: la ejecución terminó; ahora corresponde juzgar sus resultados.")

corrida completa ✅ · 7.8 s
run_id: ab7241223e9441ebbfac459225d60783
LECTURA: la ejecución terminó; ahora corresponde juzgar sus resultados.


## Paso 4 · Los objetos que recibe el comité — 8 min

Gini (`2 × AUC − 1`) y KS miden ordenamiento; los deciles muestran dónde se concentran los malos.
PSI compara distribuciones (`< 0,10` = estable). El backtesting contrasta PD esperada y mora
observada (`p < 0,05` = falla). Un Gini sano y una calibración fallida no se contradicen: uno mide
ranking y la otra, nivel y forma de las PD.

`study.artifacts.get(dominio, clave)` no recalcula: recupera el objeto que publicó cada etapa.

| Dominio / clave | Qué contiene | Pregunta que responde |
|---|---|---|
| `scorecard / scorecard` | tramos, WoE, beta y puntos | ¿Cómo se transforma una solicitud? |
| `performance / discriminant_metrics` | AUC, Gini y KS | ¿Ordena correctamente? |
| `performance / performance_table` | deciles, mora y lift | ¿Dónde concentra el riesgo? |
| `stability / stability_metrics` | PSI y banda | ¿Cambió la población? |
| `validation / calibration` | esperado, observado y pruebas | ¿Las PD tienen el nivel correcto? |
| `validation / result` | dictamen agregado | ¿Qué concluye la validación técnica? |

Los `.copy()` protegen los artefactos oficiales: los renombres y filtros de presentación ocurren
sobre copias, no sobre la evidencia guardada en `study`.

In [5]:
mapa_muestras = {"desarrollo": "DEV", "holdout": "HO", "oot": "OOT"}

# Cada par (dominio, clave) nombra una salida publicada por una etapa del pipeline.
scorecard = study.artifacts.get("scorecard", "scorecard")
metricas = study.artifacts.get("performance", "discriminant_metrics").copy()
deciles = study.artifacts.get("performance", "performance_table").copy()
estabilidad = study.artifacts.get("stability", "stability_metrics").copy()
calibracion = study.artifacts.get("validation", "calibration").copy()
validacion = study.artifacts.get("validation", "result")

metricas["muestra"] = metricas["partition"].map(mapa_muestras)
metricas_comite = metricas.set_index("muestra")[["n_total", "n_bad", "auc", "gini", "ks"]]
deciles_oot = deciles.loc[deciles["partition"].eq("oot"),
                          ["decile", "n_total", "n_bad", "bad_rate", "mean_pd", "lift", "ks_at_decile"]]
psi_oot = estabilidad.loc[
    estabilidad["metric"].eq("score_psi") & estabilidad["comparison"].eq("dev_vs_oot"),
    ["comparison", "value", "band", "action"],
]
calibracion_comite = calibracion[[
    "partition", "test", "n", "observed_defaults", "expected_pd",
    "observed_dr", "statistic", "p_value", "decision",
]]

print("SCORECARD · primeras 12 filas")
display(scorecard[["feature", "bin_label", "woe", "beta", "points"]].head(12).round(4))
print("\nGINI Y KS POR MUESTRA")
display(metricas_comite.round(4))
print("\nDECILES OOT")
display(deciles_oot.round(4))
print("\nPSI DEL SCORE DEV→OOT")
display(psi_oot.round(4))
print("\nCALIBRACIÓN: OBSERVADO VS ESPERADO")
display(calibracion_comite.round(4))
print("overall_status:", validacion.card.overall_status)

referencia_c5 = pd.DataFrame(
    {"Gini a mano C5": [0.7570, 0.6981, 0.6750],
     "Gini nikodym": metricas_comite.loc[["DEV", "HO", "OOT"], "gini"].values},
    index=["DEV", "HO", "OOT"],
)
print("\nMISMO DATASET, DOS PIPELINES")
display(referencia_c5.round(4))
print("Los Ginis se parecen; las variables no tienen por qué coincidir.")

gini_ho = metricas_comite.loc["HO", "gini"]
gini_oot = metricas_comite.loc["OOT", "gini"]
psi_valor = psi_oot.iloc[0]["value"]
hl_oot = calibracion.loc[
    calibracion["partition"].eq("oot") & calibracion["test"].eq("hosmer_lemeshow")
].iloc[0]
p_oot = float(hl_oot["p_value"])
p_oot_texto = "< 0,001" if p_oot < 0.001 else f"{p_oot:.3f}"

print()
print("LECTURA AUTÓNOMA")
print(f"1. Ranking: Gini HO {gini_ho:.4f} y OOT {gini_oot:.4f}; el ordenamiento se conserva en meses futuros.")
print(f"2. Población: PSI DEV→OOT {psi_valor:.4f} < 0,10; la distribución del score es estable.")
print(
    f"3. Calibración OOT: PD esperada {hl_oot['expected_pd']:.2%} vs mora observada "
    f"{hl_oot['observed_dr']:.2%}; p {p_oot_texto} y el test falla por tramos."
)
print("4. Dictamen: overall_status fail exige diagnóstico; no autoriza producción ni prueba que recalibrar δ baste.")

SCORECARD · primeras 12 filas
             feature        bin_label     woe    beta  points
0   abonos_delta_12m    (-inf, -0.21) -0.3524 -0.6644      57
1   abonos_delta_12m   [-0.21, -0.13) -0.3010 -0.6644      58
2   abonos_delta_12m   [-0.13, -0.01)  0.0388 -0.6644      65
3   abonos_delta_12m    [-0.01, 0.18)  0.1915 -0.6644      67
4   abonos_delta_12m     [0.18, 0.30)  0.2152 -0.6644      68
5   abonos_delta_12m      [0.30, inf)  1.0076 -0.6644      83
6   abonos_delta_12m          Special  0.0000 -0.6644      64
7   abonos_delta_12m          Missing -0.6849 -0.6644      51
8   antiguedad_meses    (-inf, 14.50) -0.7818 -0.5154      52
9   antiguedad_meses   [14.50, 73.50) -0.2850 -0.5154      60
10  antiguedad_meses   [73.50, 99.50)  0.2730 -0.5154      68
11  antiguedad_meses  [99.50, 117.50)  0.4418 -0.5154      70

GINI Y KS POR MUESTRA
         n_total  n_bad     auc    gini      ks
muestra                                        
DEV         3313    159  0.8915  0.7831  0.63

## Paso 5 · Leer el audit trail — 3 min

El trail es el registro cronológico de la corrida: eventos, artefactos y decisiones. No prueba por
sí solo que nadie lo editó. **Qué mirar:** el conteo prueba que existe una historia de la ejecución;
las decisiones permiten reconstruir qué regla produjo cada acción, pero todavía falta proteger el
archivo contra alteraciones.

`read_trail` convierte cada línea JSON en un `AuditEvent`. Su campo `kind` distingue inicio/fin,
artefactos y decisiones; por eso filtramos `kind == "decision"` para leer las reglas aplicadas.

In [6]:
# El trail ya fue escrito por nikodym durante la corrida válida del paso 3.
eventos = read_trail(TRAIL)
conteo_eventos = Counter(evento.kind for evento in eventos)
decisiones = [evento for evento in eventos if evento.kind == "decision"]

print(f"{len(eventos)} eventos · {len(decisiones)} decisiones")
print(conteo_eventos)
print("\nprimeras cuatro decisiones:")
for evento in decisiones[:4]:
    print(
        f"  {evento.step or '—':12s} · {evento.payload['regla']} "
        f"→ {evento.payload['accion']}"
    )
print("LECTURA: el volumen de eventos no prueba calidad; permite rastrear y auditar las reglas aplicadas.")

398 eventos · 341 decisiones
Counter({'decision': 341, 'artifact': 55, 'run_start': 1, 'run_end': 1})

primeras cuatro decisiones:
  —            · exclusion → marcar_excluido
  —            · partition_strategy → aplicar_estrategia
  —            · fuera_de_modelo → excluir_de_modelado
  —            · partition_summary → reportar_particiones
LECTURA: el volumen de eventos no prueba calidad; permite rastrear y auditar las reglas aplicadas.


## Paso 6 · Dos ataques: cadena y sello — 5 min

La librería registra; en esta versión la institución añade la cadena y custodia el sello aparte.
El ataque torpe edita un evento sin rehacer hashes: la cadena lo detecta. El prolijo edita y
recalcula toda la cola: la cadena vuelve a cuadrar y sólo el sello externo lo rechaza. Ninguna de
las dos defensas prueba que la decisión original era correcta; prueban integridad del registro.

La regla es `hash actual = SHA-256(hash anterior + evento canónico)`:

- `canonico` convierte un evento en texto estable: mismas claves, mismo orden, mismos bytes.
- `encadenar` hace que cada hash dependa del evento actual y de toda la historia anterior.
- `verificar` reconstruye la cadena y devuelve si coincide y dónde aparece la primera diferencia.
- `sello` guarda fuera de la cadena su hash terminal y cantidad de eventos: es el punto de
  comparación que un atacante no debería poder reescribir junto con el archivo.

Las funciones siguientes son una demostración institucional añadida sobre el JSONL de nikodym; no
son parte de la API de la librería en `1.11.0`. Aquí el sello queda como otro archivo visible en
`salida_c6`; en producción «externo» exige otra custodia o permisos independientes.

In [7]:
def canonico(evento):
    """Representa siempre el mismo evento con exactamente los mismos bytes."""
    return json.dumps(evento, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def encadenar(lista_eventos):
    """Liga cada evento con el hash previo; cambiar uno invalida toda la cola posterior."""
    anterior, cadena = "0" * 64, []
    for evento in lista_eventos:
        actual = hashlib.sha256((anterior + canonico(evento)).encode()).hexdigest()
        cadena.append({"evento": evento, "hash_anterior": anterior, "hash": actual})
        anterior = actual
    return cadena

def verificar(cadena):
    """Recalcula la cadena y devuelve (es_integra, primera_posicion_distinta)."""
    rehecha = encadenar([eslabon["evento"] for eslabon in cadena])
    diferencias = [i for i, (a, b) in enumerate(zip(cadena, rehecha)) if a["hash"] != b["hash"]]
    return not diferencias, (diferencias[0] if diferencias else None)

eventos_json = [evento.model_dump(mode="json") for evento in eventos]
cadena_original = encadenar(eventos_json)

# El sello se guarda aparte de la historia que protege.
sello = {"hash_terminal": cadena_original[-1]["hash"], "n_eventos": len(cadena_original)}
(SALIDA / "trail_encadenado.json").write_text(
    json.dumps(cadena_original, ensure_ascii=False, indent=2), encoding="utf-8"
)
(SALIDA / "sello_externo.json").write_text(
    json.dumps(sello, ensure_ascii=False, indent=2), encoding="utf-8"
)

# Ataque torpe: cambia el contenido, pero deja los hashes viejos.
ataque_torpe = copy.deepcopy(cadena_original)
posicion = next(i for i, e in enumerate(ataque_torpe) if e["evento"]["kind"] == "decision")
ataque_torpe[posicion]["evento"]["payload"]["accion"] = "aprobar_sin_control"
integra_torpe, primer_error = verificar(ataque_torpe)

# Ataque prolijo: recalcula todos los hashes desde el evento adulterado.
ataque_prolijo = encadenar([e["evento"] for e in ataque_torpe])
integra_prolijo, _ = verificar(ataque_prolijo)

# La cadena adulterada cuadra consigo misma; el sello externo conserva el final original.
sello_acepta = (
    ataque_prolijo[-1]["hash"] == sello["hash_terminal"]
    and len(ataque_prolijo) == sello["n_eventos"]
)

assert not integra_torpe and integra_prolijo and not sello_acepta
print(f"Ataque torpe · cadena: ❌ RECHAZADO en evento {primer_error + 1}")
print("Ataque prolijo · cadena: ✅ ÍNTEGRA")
print("Ataque prolijo · sello externo: ❌ RECHAZADO")
print("LECTURA: la cadena detecta una edición simple; el sello externo detecta una historia rehecha.")

Ataque torpe · cadena: ❌ RECHAZADO en evento 2
Ataque prolijo · cadena: ✅ ÍNTEGRA
Ataque prolijo · sello externo: ❌ RECHAZADO
LECTURA: la cadena detecta una edición simple; el sello externo detecta una historia rehecha.


## Paso 7 · Lineage y reproducibilidad — 2 min

El lineage es la ficha de procedencia de la corrida; no duplica la matriz ni el modelo.

| Campo | Qué identifica | Qué debería ocurrir al repetir |
|---|---|---|
| `run_id` | esta ejecución concreta | cambia |
| `config_hash` | la receta completa | se mantiene si no cambian decisiones |
| `data_hash` | los datos efectivos | se mantiene si no cambia la matriz |
| `root_seed` | la fuente de aleatoriedad | se mantiene |
| `git_sha` / `git_dirty` | estado del código disponible | documenta el entorno; no mide calidad |

**Qué mirar:** reproducir no significa obtener el mismo `run_id`, sino poder explicar el resultado
con los mismos datos, configuración, semilla y versiones. La segunda corrida queda asíncrona.

In [8]:
bundle = study.lineage_bundle()
lineage = bundle.model_dump(mode="json")
for clave in ("run_id", "config_hash", "data_hash", "root_seed", "git_sha", "git_dirty"):
    if clave in lineage:
        print(f"{clave:18s} {lineage[clave]}")
print(f"library_versions   {len(lineage.get('library_versions', {}))} paquetes registrados")
print("LECTURA: estos identificadores comprueban si dos documentos salieron de los mismos datos y config.")

config_hash        8bdf3d7bad0a37e4871f5de7fd8fef9c47ba55f199fa52435e9b3cfc13f7a9ba
data_hash          7e3b19ce98fe50a7f0af1c6659b4ed4f7ccd24b9f1c13835c48d1f23261b700d
root_seed          20240706
git_sha            None
git_dirty          False
library_versions   5 paquetes registrados
LECTURA: estos identificadores comprueban si dos documentos salieron de los mismos datos y config.


## Paso 8 · El model card — 2 min

Éste es el detalle que vale la clase: la ficha combina alcance humano con identidad automática y
declara incluso las limitaciones del entorno que la herramienta no puede resolver por sí sola.

- **Declarado por personas:** propósito, cartera, responsable, supuestos, limitaciones y revisión.
- **Ligado automáticamente a la corrida:** `run_id`, hashes, métricas, decisiones y advertencias
  de reproducibilidad.

`GovernanceConfig` valida las declaraciones institucionales; `ModelCardBuilder` las une con la
evidencia del `study` y del trail. Luego guardamos la misma ficha como JSON auditable y Markdown
legible: no son dos fichas distintas.

**Qué mirar:** propósito y limitaciones propias son juicio humano; hashes, decisiones y advertencias
del entorno quedan ligados automáticamente al `run_id`.

In [9]:
from nikodym.governance import GovernanceConfig, ModelCardBuilder

gcfg = GovernanceConfig.model_validate(cfg["governance"])
model_card = ModelCardBuilder(gcfg).build(study, trail_path=TRAIL)
(SALIDA / "model_card_c6.json").write_text(model_card.to_json(), encoding="utf-8")
(SALIDA / "model_card_c6.md").write_text(model_card.to_markdown(), encoding="utf-8")

print("run_id:      ", model_card.run_id)
print("config_hash: ", model_card.config_hash)
print("data_hash:   ", model_card.data_hash)
print("decisiones:  ", len(model_card.decisions))
print("revisión:    ", model_card.review_date, "→", model_card.next_review_date)
print("\nlimitaciones declaradas y automáticas:")
for limitacion in model_card.limitations:
    print(" ·", limitacion)
print("LECTURA: una ficha trazable separa lo declarado por el modelador de lo comprobado por el entorno.")

run_id:       ab7241223e9441ebbfac459225d60783
config_hash:  8bdf3d7bad0a37e4871f5de7fd8fef9c47ba55f199fa52435e9b3cfc13f7a9ba
data_hash:    7e3b19ce98fe50a7f0af1c6659b4ed4f7ccd24b9f1c13835c48d1f23261b700d
decisiones:   341
revisión:     2026-09-16 19:11:50.847678+00:00 → 2027-09-16 19:11:50.847678+00:00

limitaciones declaradas y automáticas:
 · Modelado sobre aprobados: no observa las rechazadas.
 · git no disponible: la corrida no tiene SHA de origen
 · lineage parcial: sin git SHA
 · lineage parcial: sin hash de uv.lock
LECTURA: una ficha trazable separa lo declarado por el modelador de lo comprobado por el entorno.


## Paso 9 · Informe y resumen ejecutivo — 1 min

El informe ya fue generado por la corrida del paso 3; esta celda no vuelve a modelar. La corrida
deja dos versiones del mismo documento: `scorecard_report.html` para leer y `scorecard_report.qmd`,
la fuente en Markdown que un analista edita. Leemos la fuente: cada línea `## ` es un capítulo.
Comprobamos que estén los ocho capítulos, los tres anexos y las tablas de esta ejecución, y mostramos
el inicio del resumen para juzgarlo, no para reemplazarlo.

El documento generado es una base editable y trazable, no la decisión del comité. La prueba final:
si el resumen no dice qué decidir, es un abstract. **Qué mirar:** el informe reúne la evidencia de
la misma corrida; recomendación, límites de uso, usos no previstos y gatillos todavía requieren
autoría y aprobación humana.

In [10]:
ruta_qmd = SALIDA / "scorecard_report.qmd"
ruta_html = SALIDA / "scorecard_report.html"
lineas = ruta_qmd.read_text(encoding="utf-8").splitlines()

# Cada «## » de la fuente Markdown es un capítulo del informe; cada <table> del HTML, una tabla.
capitulos = [linea[3:].strip() for linea in lineas if linea.startswith("## ")]
secciones = [c for c in capitulos if not c.startswith("Anexo")]
anexos = [c for c in capitulos if c.startswith("Anexo")]
numero_tablas = ruta_html.read_text(encoding="utf-8").count("<table")

assert len(secciones) == 8, secciones
assert len(anexos) == 3, anexos
print(f"informe: {len(secciones)} secciones + {len(anexos)} anexos · "
      f"{numero_tablas} tablas · HTML de {ruta_html.stat().st_size / 1024:.1f} KiB")
for titulo in secciones + anexos:
    print(" ·", titulo)

# El resumen generado: desde su título hasta el capítulo siguiente.
inicio = lineas.index("## Resumen ejecutivo") + 1
fin = next(i for i in range(inicio, len(lineas)) if lineas[i].startswith("## "))
print("\nRESUMEN GENERADO (primeras líneas)")
print("\n".join(linea for linea in lineas[inicio:fin] if linea.strip())[:900])

print("\nEXPEDIENTE")
print("Generado: corrida, métricas, scorecard, trail, lineage, model card e informe base.")
print("Pendiente del modelador: recomendación, límites propios, usos no previstos y gatillos.")
print("\nARCHIVOS GENERADOS EN salida_c6/")
for archivo in sorted(ruta.name for ruta in SALIDA.iterdir() if ruta.is_file()):
    print(" ·", archivo)
print("LECTURA FINAL: reproducible y documentado no significa aprobado; el rojo de calibración sigue abierto.")

informe: 8 secciones + 3 anexos · 126 tablas · HTML de 613.5 KiB
 · Resumen ejecutivo
 · 1 Introducción
 · 2 Contexto del modelo y de la cartera
 · 3 Metodología
 · 4 Resultados
 · 5 Validación formal
 · 6 Conclusiones y recomendación
 · 7 Limitaciones y supuestos
 · Anexo A — Lineage y reproducibilidad
 · Anexo B — Tablas detalladas
 · Anexo C — Parámetros completos

RESUMEN GENERADO (primeras líneas)
::: {.callout-important title="POR COMPLETAR — Veredicto de validación"}
Marque el veredicto: **Aprobado** · **Aprobado con observaciones** · **Rechazado**.
Campo estructural: lo marca y firma el validador independiente; no es un resultado calculado por el motor.
:::
| Métrica | Alcance | Valor | Estado |
| --- | --- | --- | --- |
| AUC | Desarrollo | 0,8915 | Sin alertas |
| Gini | Desarrollo | 0,7831 | Sin alertas |
| KS | Desarrollo | 0,6354 | Sin alertas |
| AUC | Holdout | 0,8474 | Sin alertas |
| Gini | Holdout | 0,6949 | Sin alertas |
| KS | Holdout | 0,5671 | Sin alertas |
| AUC 